# Hansen Ch.4 习题解答（计算部分）

**Chapter 4 Least Squares Regression**

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch04_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：理论摘要 + **Exercise 4.24–4.26** 数值计算。

> **写给只学过李子奈/陈强的同学：** 本章把第 3 章那个"代数对象"$\hat\beta$ 升级成"随机变量"，研究它的无偏性、方差、效率。**全章的纲是夹心方差**：
> $$\mathrm{var}(\hat\beta\mid X)=\underbrace{(X'X)^{-1}}_{\text{面包}}\underbrace{X'\Omega X}_{\text{肉}}\underbrace{(X'X)^{-1}}_{\text{面包}}.$$
> **同方差**让肉 $=\sigma^2X'X$、消掉一个面包 → 经典公式 $\sigma^2(X'X)^{-1}$；**异方差**时肉不能消，必须单独估——所有稳健 SE（HC0–HC3）差别只在"肉里每项的权重"：
> HC0 权重 1，HC1 乘 $n/(n-k)$，HC2 用 $(1-h_{ii})^{-1}$（同方差下无偏），HC3 用 $(1-h_{ii})^{-2}$（保守，最大）。本章数值计算就是要看这些 SE 在真实数据上差多少。


## 公共函数：OLS + HC0–HC3（夹心方差）

下面函数对同一组 $(\hat\beta,\hat e,h_{ii})$ 计算**五种**方差估计，差别只在"肉"$\sum X_iX_i'\hat e_i^2\,c_i$ 的权重 $c_i$：

- **同方差**：$s^2(X'X)^{-1}$，$s^2=\sum\hat e_i^2/(n-k)$（李子奈默认）。
- **HC0**：$c_i=1$（White，原始稳健 SE）。
- **HC1**：HC0 $\times n/(n-k)$（Stata `robust` 默认）。
- **HC2**：$c_i=(1-h_{ii})^{-1}$（同方差下无偏）。
- **HC3**：$c_i=(1-h_{ii})^{-2}$（保守，SE 最大）。

由 Ex 4.10，应有 $\mathrm{SE}_{HC0}\le\mathrm{SE}_{HC1}\le\mathrm{SE}_{HC2}\le\mathrm{SE}_{HC3}$。


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

def ols_hc(y, X):
    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    XXinv = np.linalg.inv(X.T @ X)
    h = np.sum(X * (X @ XXinv), axis=1)
    def sand(scale):
        u = X * (scale * e)[:, None]
        return XXinv @ (u.T @ u) @ XXinv
    Vhom = XXinv * (np.sum(e**2) / (n - k))
    V0 = sand(np.ones(n))
    V1 = V0 * (n / (n - k))
    V2 = sand(1.0 / np.sqrt(np.clip(1 - h, 1e-12, None)))
    V3 = sand(1.0 / np.clip(1 - h, 1e-12, None))
    se = lambda V: np.sqrt(np.diag(V))
    return dict(beta=beta, e=e, n=n, k=k,
                se_hom=se(Vhom), se_HC0=se(V0), se_HC1=se(V1),
                se_HC2=se(V2), se_HC3=se(V3), V_HC3=V3)

CPS = Path("../../hansen/econometrics/data/cps09mar/cps09mar.xlsx")
if not CPS.exists():
    CPS = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx")
df = pd.read_excel(CPS)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"] / (df["hours"] * df["week"]))
df["exp2"] = (df["experience"] ** 2) / 100
print("CPS n=", len(df))


## Exercise 4.24：方程 (3.49) 多种 SE

In [ ]:
mask = (df.race==4)&(df.marital==7)&(df.female==0)&(df.experience<45)
s = df.loc[mask]
y = s.lwage.to_numpy(float)
X = np.c_[s.education, s.experience, s.exp2, np.ones(len(s))]
names = ["education","experience","exp2/100","intercept"]
r = ols_hc(y, X)
tab = pd.DataFrame({
    "beta": r["beta"], "SE_hom": r["se_hom"], "HC0": r["se_HC0"],
    "HC1": r["se_HC1"], "HC2": r["se_HC2"], "HC3": r["se_HC3"],
}, index=names)
print("n=", r["n"])
tab


## Exercise 4.25：白人男性西班牙裔，HC3

In [ ]:
m = (df.race==1)&(df.female==0)&(df.hisp==1)
s = df.loc[m].copy()
s["married"] = s.marital.isin([1,2,3]).astype(float)
s["wid_div"] = s.marital.isin([4,5]).astype(float)
s["separated"] = (s.marital==6).astype(float)
s["NE"] = (s.region==1).astype(float)
s["South"] = (s.region==3).astype(float)
s["West"] = (s.region==4).astype(float)
y = s.lwage.to_numpy(float)
X = np.c_[s.education,s.experience,s.exp2,s.NE,s.South,s.West,s.married,s.wid_div,s.separated,np.ones(len(s))]
names = ["education","experience","exp2/100","NE","South","West","married","wid_div","separated","intercept"]
r = ols_hc(y, X)
pd.DataFrame({"beta": r["beta"], "HC3": r["se_HC3"]}, index=names)


## Exercise 4.26：DDK2011 tracking + 聚类 SE

In [ ]:
DDK = Path("../../hansen/econometrics/data/DDK2011/DDK2011.xlsx")
if not DDK.exists():
    DDK = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/DDK2011/DDK2011.xlsx")
ddk = pd.read_excel(DDK)
for c in ddk.columns:
    ddk[c] = pd.to_numeric(ddk[c], errors="coerce")
ts = ddk["totalscore"]
ddk["ystd"] = (ts - ts.mean()) / ts.std()

# baseline: tracking only
d0 = ddk[["ystd","tracking"]].dropna()
X0 = np.c_[d0.tracking.values, np.ones(len(d0))]
b0 = np.linalg.lstsq(X0, d0.ystd.values, rcond=None)[0]
print("tracking only (n=%d): intercept=%.3f, tracking=%.3f" % (len(d0), b0[1], b0[0]))

d = ddk[["ystd","tracking","agetest","girl","etpteacher","percentile","schoolid"]].dropna()
y = d.ystd.values
X = np.c_[d.tracking, d.agetest, d.girl, d.etpteacher, d.percentile, np.ones(len(d))]
names = ["tracking","age","girl","etpteacher","percentile","intercept"]
beta = np.linalg.lstsq(X, y, rcond=None)[0]
e = y - X @ beta
XXinv = np.linalg.inv(X.T @ X)
u = X * e[:, None]
Vhc = XXinv @ (u.T @ u) @ XXinv
meat = np.zeros((X.shape[1], X.shape[1]))
for g in np.unique(d.schoolid.values):
    idx = d.schoolid.values == g
    sc = X[idx].T @ e[idx]
    meat += np.outer(sc, sc)
Vcl = XXinv @ meat @ XXinv
out = pd.DataFrame({
    "beta": beta,
    "SE_robust": np.sqrt(np.diag(Vhc)),
    "SE_cluster": np.sqrt(np.diag(Vcl)),
}, index=names)
out["ratio"] = out["SE_cluster"] / out["SE_robust"]
print("n=", len(d), "schools=", d.schoolid.nunique())
out


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch04 的夹心方差、HC0–HC3 排序、GLS 有效性、OLS/GLS 协方差、短回归 $s^2$ 偏差。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(4)

# 设定：异方差误差，条件于 X 做蒙特卡洛
n, k, reps = 200, 4, 40000
Xc = rng.standard_normal((n, k-1))
X = np.c_[Xc, np.ones(n)]
beta = np.array([1.0, -0.5, 0.7, 2.0])
sig2 = 0.5 + 0.8 * Xc[:, 0]**2                                  # 异方差 σ_i²
D = np.diag(sig2); Om = D
XtXinv = np.linalg.inv(X.T @ X)
Vsand = XtXinv @ (X.T @ Om @ X) @ XtXinv                        # 夹心方差 (4.16)
Omi = np.diag(1 / sig2)
Vgls = np.linalg.inv(X.T @ Omi @ X)                             # GLS 渐近方差 (4.19)

Bh = np.zeros((reps, k)); Bg = np.zeros((reps, k))
for r in range(reps):
    e = rng.standard_normal(n) * np.sqrt(sig2)
    Y = X @ beta + e
    Bh[r] = XtXinv @ (X.T @ Y)                                  # OLS
    Bg[r] = np.linalg.inv(X.T @ Omi @ X) @ (X.T @ (Omi @ Y))    # GLS (已知 Ω)
bh, bg = Bh - beta, Bg - beta

print("[4.16] MC var(OLS) :", np.round(np.diag(np.cov(bh.T)), 4))
print("       夹心公式   :", np.round(np.diag(Vsand), 4))
print("[4.6]  GLS 更有效 (V_OLS - V_GLS 半正定):",
      np.all(np.linalg.eigvalsh(Vsand - Vgls) > -1e-9))
cov_og = np.cov(bh.T, bg.T)[:k, k:]
print("[4.20] cov(OLS,GLS)=var(GLS):", np.allclose(cov_og, Vgls, atol=1e-2))
print("       var(OLS-GLS)=var(OLS)-var(GLS):",
      np.allclose(np.cov((bh - bg).T), Vsand - Vgls, atol=1e-2))

# Ex 4.10: HC0 ≤ HC2 ≤ HC3（同一组残差，只换“肉”的权重）
Y = X @ beta + rng.standard_normal(n) * np.sqrt(sig2)
beta_hat = XtXinv @ (X.T @ Y)
e = Y - X @ beta_hat
h = np.sum(X * (X @ XtXinv), axis=1)

def Vfrom(w):                                                   # w 为逐观测权重
    u = X * (w * e)[:, None]
    return XtXinv @ (u.T @ u) @ XtXinv

V0 = Vfrom(np.ones(n))                                          # HC0
V2 = Vfrom(1 / np.sqrt(np.clip(1 - h, 1e-12, None)))            # HC2 权重 (1-h)^-1
V3 = Vfrom(1 / np.clip(1 - h, 1e-12, None))                     # HC3 权重 (1-h)^-2
print("[4.10] HC0≤HC2≤HC3 (半正定差):",
      np.all(np.linalg.eigvalsh(V2 - V0) > -1e-9) and
      np.all(np.linalg.eigvalsh(V3 - V2) > -1e-9))

# Ex 4.18: 短回归 s² 高估 σ²（约束为真时才无偏，需 df=n-k+q）
X1 = np.c_[Xc[:, :2], np.ones(n)]
X2 = Xc[:, 2:3]
b2 = np.array([0.8]); s0 = 1.0; k1 = X1.shape[1]
ss = []
for r in range(20000):
    e = rng.standard_normal(n) * s0
    Y = X1 @ np.array([1.0, -0.5, 2.0]) + X2 @ b2 + e
    eh = Y - X1 @ np.linalg.solve(X1.T @ X1, X1.T @ Y)
    ss.append((eh @ eh) / (n - k1))
print(f"[4.18] 短回归 s² 均值={np.mean(ss):.4f} > σ²={s0} (OVB 使 SSE 上升)")
